# Load CMIP6 Data with Intake ESM

[Intake-ESM](https://intake-esm.readthedocs.io/en/latest/) provides a higher-level interface for searching and loading Earth System Model archives such as CMIP6. It parses an [ESM Collection Spec](https://github.com/NCAR/esm-collection-spec/) (a `.json` catalog) into an [intake](https://intake.readthedocs.io/en/latest) catalog, then uses xarray to open, concatenate, and merge query results into aggregated datasets.

This notebook reproduces the Pangeo gallery example, loading ocean dissolved-oxygen (`o2`) output from the Google Cloud CMIP6 archive.

> **Note on versions.** This example was written in 2020. Recent intake-esm releases (>=2021.x) renamed some keyword arguments. If `zarr_kwargs={'consolidated': True}` raises a deprecation/error, use `xarray_open_kwargs={'consolidated': True}` instead (see the loading cell below).

In [35]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
xr.set_options(display_style='html')
import intake
%matplotlib inline

## Open the catalog

The collection spec is a `.json` file. We open it with `intake.open_esm_datastore`. Displaying the object shows how many datasets and assets the catalog contains, plus the unique values available for each search facet.

In [2]:
cat_url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(cat_url)
col

,unique
activity_id,18
institution_id,36
source_id,88
experiment_id,170
member_id,657
table_id,37
variable_id,700
grid_label,10
zstore,514818
dcpp_init_year,61


## Search the collection

Use `.search()` to filter the catalog by facets (experiment, table, variable, grid, etc.). The result's `.df` attribute is a pandas DataFrame of matching assets.

Here we request annual ocean (`Oyr`) dissolved oxygen (`o2`) on the regridded grid (`gr`) for the `ssp585` experiment.

In [24]:
cat = col.search(experiment_id='ssp585', table_id='Oyr', variable_id='o2',
                 grid_label='gr', member_id='r1i1p1f1')
cat.df

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
0,ScenarioMIP,NOAA-GFDL,GFDL-ESM4,ssp585,r1i1p1f1,Oyr,o2,gr,gs://cmip6/CMIP6/ScenarioMIP/NOAA-GFDL/GFDL-ES...,<NA>,20180701
1,ScenarioMIP,NCC,NorESM2-MM,ssp585,r1i1p1f1,Oyr,o2,gr,gs://cmip6/CMIP6/ScenarioMIP/NCC/NorESM2-MM/ss...,<NA>,20191108
2,ScenarioMIP,NCC,NorESM2-LM,ssp585,r1i1p1f1,Oyr,o2,gr,gs://cmip6/CMIP6/ScenarioMIP/NCC/NorESM2-LM/ss...,<NA>,20191108


## Load into xarray

`to_dataset_dict()` opens each matching Zarr store and aggregates results into a dictionary of xarray datasets. The keys are built from:

```
activity_id.institution_id.source_id.experiment_id.table_id.grid_label
```

In [ ]:
# Original (2020) syntax:
# dset_dict = cat.to_dataset_dict(zarr_kwargs={'consolidated': True})

# storage_options={'token': 'anon'} allows unauthenticated access to the public GCS bucket (needed on Binder)
dset_dict = cat.to_dataset_dict(
    xarray_open_kwargs={'consolidated': True},
    storage_options={'token': 'anon'}
)

list(dset_dict.keys())

## Inspect a dataset

Pick one aggregated dataset by key. Note the `member_id` dimension: intake-esm has concatenated all ensemble members into a single dataset, and the data variables are lazy Dask arrays (nothing is loaded into memory until computed).

In [31]:
ds = dset_dict["ScenarioMIP.NOAA-GFDL.GFDL-ESM4.ssp585.Oyr.gr"]
ds

<xarray.Dataset> Size: 780MB
Dimensions:         (member_id: 1, dcpp_init_year: 1, time: 86, lev: 35,
                     lat: 180, lon: 360, bnds: 2)
Coordinates:
  * member_id       (member_id) object 8B 'r1i1p1f1'
  * dcpp_init_year  (dcpp_init_year) float64 8B nan
  * time            (time) object 688B 2015-07-02 12:00:00 ... 2100-07-02 12:...
  * lev             (lev) float64 280B 2.5 10.0 20.0 ... 5.5e+03 6e+03 6.5e+03
  * lat             (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
  * lon             (lon) float64 3kB 0.5 1.5 2.5 3.5 ... 357.5 358.5 359.5
    lat_bnds        (lat, bnds) float64 3kB dask.array<chunksize=(180, 2), meta=np.ndarray>
    lev_bnds        (lev, bnds) float64 560B dask.array<chunksize=(35, 2), meta=np.ndarray>
    lon_bnds        (lon, bnds) float64 6kB dask.array<chunksize=(360, 2), meta=np.ndarray>
    time_bnds       (time, bnds) object 1kB dask.array<chunksize=(86, 2), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    o2              (member_id, dcpp_init_year, time, lev, lat, lon) float32 780MB dask.array<chunksize=(1, 1, 14, 35, 180, 360), meta=np.ndarray>
Attributes: (12/60)
    Conventions:                      CF-1.7 CMIP-6.0 UGRID-1.0
    activity_id:                      ScenarioMIP
    branch_method:                    standard
    branch_time_in_child:             60225.0
    branch_time_in_parent:            60225.0
    comment:                          <null ref>
    ...                               ...
    intake_esm_attrs:variable_id:     o2
    intake_esm_attrs:grid_label:      gr
    intake_esm_attrs:zstore:          gs://cmip6/CMIP6/ScenarioMIP/NOAA-GFDL/...
    intake_esm_attrs:version:         20180701
    intake_esm_attrs:_data_format_:   zarr
    intake_esm_dataset_key:           ScenarioMIP.NOAA-GFDL.GFDL-ESM4.ssp585....

### Quick example: plot a surface oxygen map

A small addition beyond the original gallery notebook — select the first ensemble member, the surface level, and the last time step, then plot. This triggers a Dask computation for just that 2D slice.

In [ ]:
import hvplot.xarray

surface_o2 = ds["o2"].isel(member_id=0, dcpp_init_year=0, time=-1, lev=0)
surface_o2_180 = surface_o2.assign_coords(
    lon=(surface_o2.lon + 180) % 360 - 180
).sortby("lon")

# Subset to Mediterranean BEFORE plotting — avoids extent/projection conflicts
med = surface_o2_180.sel(lat=slice(28, 48), lon=slice(-6, 42))

med.hvplot.image(
    x="lon", y="lat",
    geo=True,
    tiles="OSM",
    alpha=0.7,
    cmap="Spectral_r",
    clabel="Dissolved oxygen (mol m⁻³)",
    title="SSP5-8.5 — surface dissolved oxygen, 2100 (Mediterranean)",
    width=700,
    height=450,
)